# AMR-RL: Exploratory Analysis

Notebook for understanding the environment dynamics before training.
Covers:
1. PK/PD dynamics — concentration curves and kill rates
2. Environment sanity checks — dose response, resistance emergence
3. Baseline policy comparison (no training needed)
4. Reward decomposition across dose strategies
5. OOD profile visualisation


In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

%matplotlib inline
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
})
print("Imports ok")


## 1. PK/PD Dynamics

Visualize drug concentration curves and Hill kill rate for each drug profile.

In [ ]:
from simulator.pkpd.pharmacokinetics import PKParams, PDParams, PKPDModel
from simulator.envs.amr_env import DRUG_PROFILES

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

t_hours = np.linspace(0, 24, 200)
doses = [200.0, 400.0]

for ax, (drug, (pk, pd)) in zip(axes, DRUG_PROFILES.items()):
    for dose in doses:
        c = np.array([pk.concentration_at(dose, t) for t in t_hours])
        ax.plot(t_hours, c, label=f"{dose:.0f} mg")
    ax.axhline(pd.ec50, color="red", linestyle="--", linewidth=1, label=f"EC50={pd.ec50}")
    ax.set_title(drug)
    ax.set_xlabel("Hours")
    ax.set_ylabel("Concentration (mg/L)")
    ax.legend(fontsize=9)

fig.suptitle("Plasma concentration curves (single dose)", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Hill kill rate curves
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
c_range = np.linspace(0, 3, 300)

for ax, (drug, (pk, pd)) in zip(axes, DRUG_PROFILES.items()):
    kill = np.array([pd.kill_rate(c) for c in c_range])
    ax.plot(c_range, kill, color="#534AB7", linewidth=2)
    ax.axvline(pd.ec50, color="red", linestyle="--", linewidth=1, label=f"EC50={pd.ec50}")
    ax.axhline(pd.e_max / 2, color="gray", linestyle=":", linewidth=1, label="E_max/2")
    ax.set_title(drug)
    ax.set_xlabel("Concentration (mg/L)")
    ax.set_ylabel("Kill rate (h⁻¹)")
    ax.legend(fontsize=9)

fig.suptitle("PD kill rate — Hill equation", y=1.02)
plt.tight_layout()
plt.show()


## 2. Environment Dose-Response

Run the full env at each dose level and observe bacterial load trajectory.

In [ ]:
from simulator.envs.amr_env import AMREnv, DOSE_LEVELS

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = plt.cm.viridis(np.linspace(0, 1, len(DOSE_LEVELS)))

# Left: load traces
ax = axes[0]
for i, dose in enumerate(DOSE_LEVELS):
    env = AMREnv(seed=0)
    env.reset()
    loads = [env._load]
    for _ in range(14):
        _, _, term, trunc, info = env.step(i)
        loads.append(info["bacterial_load"])
        if term or trunc:
            break
    env.close()
    days = np.arange(len(loads))
    ax.plot(days, np.log10(np.maximum(loads, 1)), color=colors[i],
            label=f"{dose:.0f} mg", linewidth=2)

ax.axhline(np.log10(1e3), color="red", linestyle="--", linewidth=1, label="Target (10³)")
ax.set_xlabel("Day"); ax.set_ylabel("log₁₀ CFU/mL")
ax.set_title("Bacterial load by dose level")
ax.legend(fontsize=9)

# Right: resistance traces
ax = axes[1]
for i, dose in enumerate(DOSE_LEVELS):
    env = AMREnv(seed=42)
    env.reset()
    resistances = [env._resistance]
    for _ in range(14):
        _, _, term, trunc, info = env.step(i)
        resistances.append(info["resistance_level"])
        if term or trunc:
            break
    env.close()
    ax.plot(np.arange(len(resistances)), resistances, color=colors[i],
            label=f"{dose:.0f} mg", linewidth=2)

ax.set_xlabel("Day"); ax.set_ylabel("Resistance level (0–4)")
ax.set_title("Resistance emergence by dose level")
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()


## 3. Baseline Policy Comparison

Compare cycling, bandit, max dose, and no dose across 100 episodes each.

In [ ]:
from baselines.baselines import CyclingHeuristic, ContextualBanditPolicy, MaxDosePolicy, NoDosePolicy
from evaluation.metrics.eval_metrics import evaluate_policy, compare_policies

env = AMREnv(seed=0)

policies = {
    "cycling":  CyclingHeuristic(period=3),
    "bandit":   ContextualBanditPolicy(),
    "max_dose": MaxDosePolicy(),
    "no_dose":  NoDosePolicy(),
}

results = {}
for name, policy in policies.items():
    print(f"  Evaluating {name}...", end=" ", flush=True)
    results[name] = evaluate_policy(policy, env, n_episodes=100, policy_name=name)
    print(f"clearance={results[name].clearance_rate:.0%}, mean_r={results[name].mean_reward:.2f}")

env.close()
print()
compare_policies(results)


In [ ]:
# Reward distributions
fig, ax = plt.subplots(figsize=(9, 4))

palette = {"cycling": "#D85A30", "bandit": "#BA7517", "max_dose": "#888780", "no_dose": "#E24B4A"}

for name, res in results.items():
    rewards = [r.total_reward for r in res.records]
    ax.hist(rewards, bins=20, alpha=0.6, label=name, color=palette.get(name, "#333"))

ax.set_xlabel("Episode reward")
ax.set_ylabel("Count")
ax.set_title("Reward distribution by baseline policy (100 episodes each)")
ax.legend()
plt.tight_layout()
plt.show()


## 4. Reward Decomposition

Break down which reward components matter most at different dose levels.

In [ ]:
from simulator.reward.reward_fn import RewardFunction, RewardConfig

rf = RewardFunction()
components_by_dose = {d: {"load": [], "dose": [], "resistance": [], "msw": []}
                      for d in DOSE_LEVELS}

for d_idx, dose in enumerate(DOSE_LEVELS):
    env = AMREnv(seed=0)
    obs, _ = env.reset()
    for _ in range(14):
        obs, _, term, trunc, info = env.step(d_idx)
        components_by_dose[dose]["load"].append(info.get("load", 0))
        components_by_dose[dose]["dose"].append(info.get("dose", 0))
        components_by_dose[dose]["resistance"].append(info.get("resistance", 0))
        components_by_dose[dose]["msw"].append(info.get("msw", 0))
        if term or trunc: break
    env.close()

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
comp_names = ["load", "dose", "resistance", "msw"]
comp_colors = ["#534AB7", "#D85A30", "#E24B4A", "#BA7517"]

for ax, comp, color in zip(axes, comp_names, comp_colors):
    means = [np.mean(components_by_dose[d][comp]) for d in DOSE_LEVELS]
    ax.bar([f"{d:.0f}" for d in DOSE_LEVELS], means, color=color, alpha=0.8)
    ax.set_title(f"{comp} component")
    ax.set_xlabel("Dose (mg)")

fig.suptitle("Mean reward component per step by dose level", y=1.02)
plt.tight_layout()
plt.show()


## 5. OOD Profile Visualisation

Inspect the held-out resistance profiles used in OOD evaluation.

In [ ]:
from evaluation.metrics.eval_metrics import generate_ood_profiles

profiles = generate_ood_profiles(n_profiles=20, seed=99)

init_resistances = [p["initial_resistance"] for p in profiles]
fitness_costs    = [p.get("fitness_cost_override", 0.08) for p in profiles]
ec50_mults       = [p.get("ec50_multiplier", 1.0) for p in profiles]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].hist(init_resistances, bins=[0, 1, 2, 3, 4], rwidth=0.7, color="#534AB7", alpha=0.8)
axes[0].set_title("Initial resistance level"); axes[0].set_xlabel("Level (0–4)")

axes[1].scatter(range(len(fitness_costs)), fitness_costs, color="#1D9E75", alpha=0.8)
axes[1].axhline(0.08, color="red", linestyle="--", label="Training default")
axes[1].set_title("Fitness cost override"); axes[1].set_xlabel("Profile index")
axes[1].legend()

axes[2].scatter(range(len(ec50_mults)), ec50_mults, color="#D85A30", alpha=0.8)
axes[2].axhline(1.0, color="red", linestyle="--", label="Baseline EC50")
axes[2].set_title("EC50 multiplier (resistance severity)"); axes[2].set_xlabel("Profile index")
axes[2].legend()

fig.suptitle("20 OOD resistance profiles", y=1.02)
plt.tight_layout()
plt.show()

print(f"\nProfiles span:")
print(f"  Initial resistance: {min(init_resistances):.0f}–{max(init_resistances):.0f}")
print(f"  Fitness cost:       {min(fitness_costs):.3f}–{max(fitness_costs):.3f}")
print(f"  EC50 multiplier:    {min(ec50_mults):.2f}–{max(ec50_mults):.2f}")


## Next steps

Once training is complete, run `scripts/evaluate.py` to generate all paper figures:

```bash
python scripts/evaluate.py \
    --policy_path checkpoints/best/best_model.zip \
    --fixed_policy_path checkpoints/fixed_ppo_static.zip \
    --training_history runs/training_history.json \
    --output_dir results/
```
